# CheXzero Backbone Verification

**Model**: CheXzero (`rajpurkarlab/CheXzero`) — CLIP pretrained on MIMIC-CXR reports  
**Arch**: Custom ViT (12 layers, 768-width, 16-px patches), CLIP contrastive training  
**Embed dim**: 768 (L2-normalized image features)  
**Input**: 3×320×320 px, CXR-specific normalization (NOT ImageNet)  
**Access**: Weights on Google Drive (public, no auth)  
**Zero-shot**: Yes — paired text encoder via CLIP

Paper: [Expert-Level Detection of Pathologies from Unannotated CXRs via Self-Supervised Learning (Nature Biomedical Engineering, 2022)](https://www.nature.com/articles/s41551-022-00936-9)

### Checkpoint download

```
https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno
```

Multiple checkpoint files are available (ensemble). Download any one (e.g. `best_64_0.0001_original_16000_0.861.pt`).

### Prerequisites

```bash
git clone https://github.com/rajpurkarlab/CheXzero /tmp/CheXzero
pip install -r /tmp/CheXzero/requirements.txt
# Place checkpoint at /tmp/CheXzero/checkpoints/chexzero_weights/best.pt
```

In [ ]:
%pip install -q torch torchvision Pillow
# openai/CLIP is also needed — install from source
%pip install -q git+https://github.com/openai/CLIP.git

import subprocess, os, sys

CHEXZERO_DIR = "/tmp/CheXzero"
if not os.path.exists(CHEXZERO_DIR):
    subprocess.run(["git", "clone", "https://github.com/rajpurkarlab/CheXzero",
                    CHEXZERO_DIR], check=True)

sys.path.insert(0, CHEXZERO_DIR)
print(os.listdir(CHEXZERO_DIR))

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Locate checkpoint

In [ ]:
import os

# Update after downloading from Google Drive
CKPT_PATH = "/tmp/CheXzero/checkpoints/chexzero_weights/best.pt"

assert os.path.exists(CKPT_PATH), (
    f"Checkpoint not found at {CKPT_PATH}.\n"
    "Download from: https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno"
)
print("Checkpoint found:", os.path.getsize(CKPT_PATH) / 1e6, "MB")

## 2. Load model

CheXzero is a custom CLIP variant: same architecture as CLIP ViT-B/32 but
re-trained end-to-end on MIMIC-CXR (image, report) pairs at 320×320 resolution.

In [ ]:
import torch
import clip  # openai/CLIP

# CheXzero uses CLIP's internal model definition but with custom weights
# Load the base CLIP architecture then override state dict
device = "cuda" if torch.cuda.is_available() else "cpu"

# The checkpoint stores the CLIP model state dict directly
ckpt = torch.load(CKPT_PATH, map_location=device)
print("Checkpoint type:", type(ckpt))
if isinstance(ckpt, dict):
    print("Keys:", list(ckpt.keys())[:5])

# CheXzero custom config: ViT, 12 layers, 768-width, 16px patches, 320px resolution
# Build from CLIP's internal ModelDimensions
try:
    from clip.model import CLIP, build_model
    # The checkpoint might be the full state dict of a CLIP model
    state_dict = ckpt if isinstance(ckpt, dict) and "visual.conv1.weight" in ckpt else ckpt.get("state_dict", ckpt)
    model = build_model(state_dict).to(device)
    print("Loaded via build_model")
except Exception as e:
    print(f"build_model failed ({e}), trying direct load")
    # Fallback: load as pretrained CLIP then override weights
    model, preprocess_clip = clip.load("ViT-B/32", device=device, jit=False)
    model.load_state_dict(ckpt, strict=False)

model.eval()
print("Model loaded on", device)
print("Visual input resolution:", model.visual.input_resolution)

## 3. Image transform

CheXzero uses **CXR-specific normalization** (computed from the MIMIC-CXR training set), **not** ImageNet stats.
Input: 320×320, 3-channel (grayscale repeated), values in [0, 255] before normalization.

In [ ]:
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

# CXR-specific statistics from zero_shot.py in the CheXzero repo
CXR_MEAN = (101.48761 / 255.0,) * 3   # convert to [0,1] space
CXR_STD  = (83.43944  / 255.0,) * 3

INPUT_SIZE = 320

def make_transform():
    return transforms.Compose([
        transforms.Resize(INPUT_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1)),   # 1ch → 3ch
        transforms.Normalize(mean=CXR_MEAN, std=CXR_STD),
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    """Return [1, 3, 320, 320] batch tensor."""
    img = Image.open(path)
    return transform(img).unsqueeze(0).to(device)

dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)
print("dummy input:", dummy.shape)

## 4. Extract image embeddings

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    img_feats = model.encode_image(dummy)   # [B, 768]
    img_feats = F.normalize(img_feats, dim=-1)   # L2-normalize (CLIP convention)

print("image features:", img_feats.shape)   # [1, 768]
print("L2 norm:", img_feats.norm(dim=-1).item())

## Segmentation mode — patch tokens

For dense prediction, extract the per-patch token sequence before the final linear
projection using a forward hook on `visual.ln_post`. The 320×320 input with 16-px
patches gives a **20×20** patch grid (400 tokens, 768-dim).

In [ ]:
import math

# Capture pre-projection token sequence via a forward hook on the final LayerNorm.
# CLIP applies a linear projection at the very end of encode_image; hooking ln_post
# gives the full [CLS, patch_1, ..., patch_N] sequence before that projection.
pre_proj = {}
def _hook(module, inp, out):
    pre_proj["tokens"] = out

handle = model.visual.ln_post.register_forward_hook(_hook)
with torch.no_grad():
    _ = model.encode_image(dummy)
handle.remove()

all_tokens = pre_proj["tokens"]   # [B, 1 + N_patches, D] before projection
patches    = all_tokens[:, 1:]    # [B, N_patches, D] — skip CLS at index 0

B, N, D = patches.shape
H = W = int(math.sqrt(N))         # 20 for 320px / 16px-patches
assert H * W == N

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 400, 768]
print(f"spatial map:   {spatial.shape}")   # [1, 768, 20, 20]

## 5. Zero-shot: text embeddings & similarity

In [ ]:
# CheXzero-style prompts: pairs of positive/negative templates
# E.g., for zero-shot: use difference of positive - negative logit scores
positive_prompts = [
    "Findings consistent with pleural effusion.",
    "Cardiomegaly is present.",
    "Pneumothorax identified.",
]
negative_prompts = [
    "No pleural effusion.",
    "Heart size is normal.",
    "No pneumothorax.",
]

pos_tokens = clip.tokenize(positive_prompts).to(device)
neg_tokens = clip.tokenize(negative_prompts).to(device)

with torch.no_grad():
    pos_feats = F.normalize(model.encode_text(pos_tokens), dim=-1)  # [3, 768]
    neg_feats = F.normalize(model.encode_text(neg_tokens), dim=-1)  # [3, 768]

# Zero-shot score = sim(img, pos) - sim(img, neg)
pos_sim = (img_feats @ pos_feats.T).squeeze(0)   # [3]
neg_sim = (img_feats @ neg_feats.T).squeeze(0)   # [3]
scores  = pos_sim - neg_sim                       # [3]

labels = ["Pleural Effusion", "Cardiomegaly", "Pneumothorax"]
for lbl, s in zip(labels, scores.tolist()):
    print(f"{s:+.4f}  {lbl}  (+ means likely present)")

## Notes for RadHarmony integration

```
model_call : lambda m, x: F.normalize(m.encode_image(x), dim=-1)
pool       : None
embed_dim  : 768
transform  : Resize(320,BICUBIC) → CenterCrop(320) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(mean=101.5/255, std=83.4/255)
```

**CXR normalization** is the critical difference from other backbones — using ImageNet
mean/std will degrade performance significantly. The values (101.48/83.44) were computed
from the MIMIC-CXR training set pixel distribution.

**Ensemble**: The repo provides multiple checkpoints for ensembling. For a single-model
evaluation, use any one checkpoint. For maximum zero-shot performance, ensemble all.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Hook `model.visual.ln_post` to get `[B, 1+N, D]` pre-projection tokens
- Patch tokens: `all_tokens[:, 1:]` → `[B, 400, 768]`
- Spatial map: reshape to `[B, 768, 20, 20]` (320px / 16px-patches → 20×20)

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH)
with torch.no_grad():
    feats = F.normalize(model.encode_image(x), dim=-1)

print("features:", feats.shape)
pos_sim = (feats @ pos_feats.T).squeeze(0)
neg_sim = (feats @ neg_feats.T).squeeze(0)
for lbl, s in zip(labels, (pos_sim - neg_sim).tolist()):
    print(f"{s:+.4f}  {lbl}")

## Notes for RadHarmony integration

```
model_call : lambda m, x: F.normalize(m.encode_image(x), dim=-1)
pool       : None
embed_dim  : 768
transform  : Resize(320,BICUBIC) → CenterCrop(320) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(mean=101.5/255, std=83.4/255)
```

**CXR normalization** is the critical difference from other backbones — using ImageNet
mean/std will degrade performance significantly. The values (101.48/83.44) were computed
from the MIMIC-CXR training set pixel distribution.

**Ensemble**: The repo provides multiple checkpoints for ensembling. For a single-model
evaluation, use any one checkpoint. For maximum zero-shot performance, ensemble all.